# Día 10 · Unidad 12 — Valoración estocástica de opciones: Monte Carlo vs. Black-Scholes

**Objetivos de aprendizaje**
- Simular trayectorias de un activo mediante un movimiento browniano geométrico (GBM) con NumPy.
- Valorar una opción europea (call y put) por simulación Monte Carlo.
- Implementar la fórmula cerrada de Black-Scholes con `scipy.stats.norm` y usarla como caso de control del resultado simulado.

**Duración estimada:** 45-60 minutos.

**Requisitos previos:** `04_teoria_rentas.ipynb` de este mismo día. Se asume dominio previo de la teoría de valoración de opciones (GBM, neutralidad al riesgo, fórmula de Black-Scholes) — aquí el foco es la implementación y la validación numérica.

## 1. El modelo: movimiento browniano geométrico

Bajo la medida neutral al riesgo, el precio de un activo subyacente sin dividendos sigue:

$$S_T = S_0 \cdot \exp\left[\left(r - \tfrac{1}{2}\sigma^2\right) T + \sigma \sqrt{T}\, Z\right], \qquad Z \sim N(0, 1)$$

donde $S_0$ es el precio inicial, $r$ el tipo libre de riesgo, $\sigma$ la volatilidad anualizada y $T$ el tiempo a vencimiento (en años). Es la solución exacta de la SDE del GBM en $T$ — no hace falta simular la trayectoria completa paso a paso para el precio de una opción europea (que solo depende de $S_T$), basta con simular el valor final directamente.

In [ ]:
import numpy as np
from scipy.stats import norm

# Parametros de mercado del ejemplo (una garantia financiera ligada a un
# indice de referencia, tipico de un producto de seguro de vida unit-linked)
S0 = 100.0     # precio inicial del subyacente
K = 105.0      # precio de ejercicio (strike)
r = 0.03       # tipo libre de riesgo (anualizado)
sigma = 0.20   # volatilidad anualizada
T = 1.0        # tiempo a vencimiento, en anios

## 2. Fórmula cerrada de Black-Scholes

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2) T}{\sigma \sqrt{T}}, \qquad d_2 = d_1 - \sigma \sqrt{T}$$

$$C = S_0 \, \Phi(d_1) - K e^{-rT} \, \Phi(d_2), \qquad P = K e^{-rT} \, \Phi(-d_2) - S_0 \, \Phi(-d_1)$$

donde $\Phi$ es la función de distribución acumulada de la normal estándar (`scipy.stats.norm.cdf`).

In [ ]:
def black_scholes(S0: float, K: float, r: float, sigma: float, T: float, tipo: str = "call") -> float:
    """Precio de una opcion europea con la formula cerrada de Black-Scholes.

    Args:
        S0: precio inicial del subyacente.
        K: precio de ejercicio (strike).
        r: tipo libre de riesgo anualizado (capitalizacion continua).
        sigma: volatilidad anualizada del subyacente.
        T: tiempo a vencimiento, en anios.
        tipo: "call" o "put".

    Returns:
        float: precio de la opcion.

    Raises:
        ValueError: si `tipo` no es "call" ni "put".
    """
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if tipo == "call":
        return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif tipo == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    else:
        raise ValueError('tipo debe ser "call" o "put"')


precio_call_bs = black_scholes(S0, K, r, sigma, T, tipo="call")
precio_put_bs = black_scholes(S0, K, r, sigma, T, tipo="put")
print(f"Call (Black-Scholes): {precio_call_bs:.4f}")
print(f"Put  (Black-Scholes): {precio_put_bs:.4f}")

**Validación de la propia fórmula**: la paridad put-call es una identidad que debe cumplirse siempre, independientemente de los parámetros:

$$C - P = S_0 - K e^{-rT}$$

In [ ]:
paridad_izquierda = precio_call_bs - precio_put_bs
paridad_derecha = S0 - K * np.exp(-r * T)
print(f"C - P = {paridad_izquierda:.6f}")
print(f"S0 - K*e^(-rT) = {paridad_derecha:.6f}")
assert np.isclose(paridad_izquierda, paridad_derecha)
print("Paridad put-call verificada.")

## 3. Simulación Monte Carlo

Simulamos `n_sims` valores finales $S_T$ independientes, calculamos el *payoff* de cada uno y descontamos la media al valor presente. Fijamos la semilla (`np.random.seed`) para que el resultado sea reproducible en cualquier ejecución de este notebook.

In [ ]:
def precio_montecarlo(
    S0: float, K: float, r: float, sigma: float, T: float,
    n_sims: int, tipo: str = "call", seed: int = 42,
) -> tuple[float, float]:
    """Precio de una opcion europea por simulacion Monte Carlo (GBM en un unico paso).

    Args:
        S0: precio inicial del subyacente.
        K: precio de ejercicio (strike).
        r: tipo libre de riesgo anualizado (capitalizacion continua).
        sigma: volatilidad anualizada del subyacente.
        T: tiempo a vencimiento, en anios.
        n_sims: numero de trayectorias simuladas.
        tipo: "call" o "put".
        seed: semilla del generador aleatorio, para reproducibilidad.

    Returns:
        tuple[float, float]: (precio estimado, error estandar de la estimacion).

    Raises:
        ValueError: si `tipo` no es "call" ni "put".
    """
    if tipo not in ("call", "put"):
        raise ValueError('tipo debe ser "call" o "put"')

    rng = np.random.default_rng(seed)
    Z = rng.standard_normal(n_sims)
    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

    if tipo == "call":
        payoffs = np.maximum(S_T - K, 0.0)
    else:
        payoffs = np.maximum(K - S_T, 0.0)

    payoffs_descontados = np.exp(-r * T) * payoffs
    precio = payoffs_descontados.mean()
    error_estandar = payoffs_descontados.std(ddof=1) / np.sqrt(n_sims)
    return precio, error_estandar


precio_call_mc, error_call_mc = precio_montecarlo(S0, K, r, sigma, T, n_sims=200_000, tipo="call", seed=42)
print(f"Call (Monte Carlo, 200.000 sims): {precio_call_mc:.4f}  (error estandar: {error_call_mc:.4f})")
print(f"Call (Black-Scholes):             {precio_call_bs:.4f}")

## 4. Validación: Monte Carlo frente a Black-Scholes

Este es el ejemplo canónico de "valida siempre numéricamente" que menciona `docs/uso_ia_en_el_curso.md`: dos métodos completamente distintos (uno analítico, uno simulado) deben coincidir dentro de un margen de error razonable. La tolerancia no puede ser arbitraria: usamos un múltiplo del error estándar de la propia simulación (que ya calcula `precio_montecarlo`), porque es la medida correcta de cuánto puede desviarse una estimación Monte Carlo por puro azar de muestreo.

In [ ]:
tolerancia = 4 * error_call_mc  # ~4 errores estandar: margen amplio pero no arbitrario
diferencia = abs(precio_call_mc - precio_call_bs)

print(f"|MC - BS| = {diferencia:.4f}")
print(f"Tolerancia (4 x error estandar de la simulacion) = {tolerancia:.4f}")
assert diferencia < tolerancia, "La simulacion Monte Carlo no coincide con Black-Scholes dentro de la tolerancia esperada"
print("Validado: Monte Carlo y Black-Scholes coinciden dentro del margen de error de la simulacion.")

In [ ]:
precio_put_mc, error_put_mc = precio_montecarlo(S0, K, r, sigma, T, n_sims=200_000, tipo="put", seed=42)
tolerancia_put = 4 * error_put_mc
diferencia_put = abs(precio_put_mc - precio_put_bs)

print(f"Put (Monte Carlo, 200.000 sims): {precio_put_mc:.4f}  (error estandar: {error_put_mc:.4f})")
print(f"Put (Black-Scholes):             {precio_put_bs:.4f}")
print(f"|MC - BS| = {diferencia_put:.4f}  |  tolerancia = {tolerancia_put:.4f}")
assert diferencia_put < tolerancia_put
print("Validado tambien para la put.")

> ⚠️ **Error típico**: fijar una tolerancia numérica "a ojo" (p. ej. `assert abs(mc - bs) < 0.01`) sin relacionarla con el número de simulaciones. Con pocas simulaciones (`n_sims` bajo), el error estándar es grande y una tolerancia fija demasiado ajustada haría fallar el `assert` aunque el código esté bien; con muchas simulaciones, una tolerancia fija demasiado laxa no detectaría un error real de implementación. Basar la tolerancia en el propio error estándar de la simulación (como arriba) escala correctamente en ambos casos.

## 5. Cómo escala el error con el número de simulaciones

El error estándar de una media Monte Carlo decrece con $1/\sqrt{n\_sims}$: para reducirlo a la mitad hace falta **cuadruplicar** el número de simulaciones, no solo duplicarlo. Lo comprobamos empíricamente:

In [ ]:
for n_sims in [1_000, 10_000, 100_000, 1_000_000]:
    precio, error = precio_montecarlo(S0, K, r, sigma, T, n_sims=n_sims, tipo="call", seed=42)
    print(f"n_sims={n_sims:>9,}:  precio={precio:.4f}  error_estandar={error:.5f}  |precio-BS|={abs(precio - precio_call_bs):.5f}")

**Para ti:** cambia `sigma` a `0.40` (el doble de volatilidad) y repite la validación de la sección 4 para la call. ¿Sigue cumpliéndose el `assert`? ¿Qué le pasa al precio de la opción y a la tolerancia necesaria al subir la volatilidad?

**Para ti:** implementa `precio_montecarlo_antithetic`, una variante que use **variables antitéticas** (para cada `Z` simulado, usa también `-Z`, y promedia ambos payoffs) — es una técnica estándar de reducción de varianza. Compara el error estándar resultante con el de `precio_montecarlo` para el mismo `n_sims`.

## Resumen

- El GBM tiene solución cerrada en $T$: para el precio de una opción europea basta simular $S_T$ directamente, sin necesidad de simular la trayectoria completa paso a paso.
- Black-Scholes (fórmula cerrada, con `scipy.stats.norm.cdf`) y Monte Carlo (simulación, con semilla fija para reproducibilidad) deben coincidir dentro de un margen de error justificado por el propio error estándar de la simulación — nunca dentro de una tolerancia arbitraria sin relación con `n_sims`.
- La paridad put-call es una validación adicional, independiente de la simulación, que cualquier implementación de Black-Scholes debe superar.
- El error de una estimación Monte Carlo decrece con $1/\sqrt{n\_sims}$: cuadruplicar las simulaciones reduce el error a la mitad, no duplicarlas.
- Este es el patrón general para cualquier cálculo actuarial-financiero implementado con ayuda de una IA: cuando existe un caso de control conocido (una fórmula cerrada, una identidad como la paridad put-call, un caso límite), se usa siempre para validar el resultado numérico — nunca se da por bueno un cálculo solo porque "el código se ejecuta sin error".

**Siguiente:** `06_teoria_copulas.ipynb` — cópula gaussiana para modelar dependencia entre dos líneas de negocio.